# Part 2 — Notebook 01: Jet Clustering, Radius Dependence, & Subjet Exploration

Welcome to **Part 2** of the Experimental High-Energy Physics (HEP) Monte Carlo & Analysis Training Series!

In this notebook, you will transition from matrix-element generator truth partons to **reconstructed jets** — the observable, collimated sprays of hadrons produced when colored quarks and gluons shower and hadronize in a particle physics collision.

Using a Large Hadron Collider (LHC) $Z \to b\bar{b}$ dataset containing both detector-level (EMPFlow) and truth-level constituent 4-vectors, you will explore how sequential recombination jet algorithms (anti-$k_T$, $k_t$, Cambridge/Aachen) construct jets, compare FastJet to a pure-Python algorithm implementation, analyze the impact of the radius parameter $R$, perform $(\eta, \phi)$ event visualizations mapping all constituents and jet boundaries, and recluster large-$R$ jets into subjets to reveal internal two-prong decay structure.


## Step 1: Integrated Physics Foundations — Hadronization, Color Confinement, & Jet Algorithms

Before configuring your workspace or clustering particles, let's understand the physics of hadronic jets and sequential recombination algorithms.

### 1.1 Why Do We Need Jets?
In Quantum Chromodynamics (QCD), colored fundamental particles — **quarks** and **gluons** — carry color charge and cannot exist as isolated free particles (a phenomenon known as **color confinement**).

When a high-energy scattering process creates energetic quarks or gluons:
1. **Parton Showering**: The initial parton radiates gluons ($q \to qg$), which in turn split into quark-antiquark pairs ($g \to q\bar{q}$), producing a shower of partons.
2. **Hadronization**: As the system expands and cools, color forces bind these partons into color-singlet hadrons (pions $\pi^\pm, \pi^0$, kaons $K$, protons $p$, neutrons $n$).
3. **Collimated Spray**: Because the initial parton possessed high transverse momentum $p_T$, the resulting daughter hadrons emerge tightly collimated along the direction of the original parton.

A **jet** is an algorithmically defined proxy for the underlying parton (quark or gluon), constructed by grouping nearby final-state particles according to a mathematically reproducible rule.

---

### 1.2 Sequential Recombination Jet Algorithms
High-energy physics relies on **sequential recombination algorithms** to construct jets from a variable-length list of constituent 4-vectors. These algorithms compute two distance metrics for all particles/clusters in an event:

1. **Pair Distance** ($d_{ij}$): Distance between pseudo-jets $i$ and $j$:
   $$d_{ij} = \min\left(p_{T,i}^{2p}, p_{T,j}^{2p}\right) \frac{\Delta R_{ij}^2}{R^2}$$
2. **Beam Distance** ($d_{iB}$): Distance between pseudo-jet $i$ and the collider beam axis:
   $$d_{iB} = p_{T,i}^{2p}$$

Where $\Delta R_{ij}^2 = (\eta_i - \eta_j)^2 + (\phi_i - \phi_j)^2$ is the angular separation, $R$ is the jet radius parameter, and $p$ determines the algorithmic power:

| Algorithm Name | Power Parameter ($p$) | Distance Metric $d_{ij}$ Feature | Clustered Jet Shape Characteristics |
| :--- | :---: | :--- | :--- |
| **anti-$k_t$** | $p = -1$ | Hard particles ($p_{T,i}^{-2}$) recombine first | Forms stable, rigid, circular boundaries around hard cores |
| **$k_t$** | $p = +1$ | Soft particles ($p_{T,i}^{+2}$) recombine first | Irregular boundaries; useful for QCD subjets and clustering history |
| **Cambridge/Aachen (C/A)** | $p = 0$ | Pure angular distance $\Delta R_{ij}^2 / R^2$ | Purely geometrical proximity; ideal for boosted decay reclustering |

#### The Iterative Algorithm Loop:
1. Compute all $d_{ij}$ and $d_{iB}$ for the current list of objects.
2. Find the global minimum distance $d_{\text{min}} = \min(d_{ij}, d_{iB})$.
3. If $d_{\text{min}}$ is a pair distance $d_{ij}$, combine objects $i$ and $j$ into a single 4-vector $p_{ij}^\mu = p_i^\mu + p_j^\mu$, remove $i$ and $j$, and return to Step 1.
4. If $d_{\text{min}}$ is a beam distance $d_{iB}$, declare object $i$ a **completed jet**, remove it from the active list, and return to Step 1.
5. Stop when no active objects remain.

---

### 1.3 Radius Parameter $R$
The parameter $R$ defines the effective angular clustering radius in $(\eta, \phi)$ space:
- **Small-$R$ Jets ($R = 0.4$)**: Standard scale for resolving individual quarks, gluons, and isolated decay products while minimizing contamination from background radiation.
- **Large-$R$ Jets ($R = 0.8$ or $R = 1.0$)**: Wide scale designed to capture all collimated decay products of a high-$p_T$ boosted heavy resonance (like $Z \to b\bar{b}$ or $W \to q\bar{q}$) inside a single merged jet.

---

### 1.4 EMPFlow vs. Truth Constituents
The ROOT dataset `Zbb_RawConst.root` contains two sets of event constituents:
- **`constituents_EMPFlow_*`**: Detector-level Energy-Matched Particle Flow objects (charged tracks matched to inner detector measurements + neutral calorimeter topological clusters).
- **`constituents_Truth_*`**: Generator-level truth particles prior to detector interaction.

For particle identification, refer to the official [PDG Monte Carlo Particle Numbering Scheme](https://pdg.lbl.gov/2007/reviews/montecarlorpp.pdf).

---

> [!IMPORTANT]
> ### Self-Reflection Checkpoint 1.1
> **Conceptual Question**: Why is a jet algorithm needed instead of simply calling every final-state particle a jet?

<details>
<summary>Click to show Checkpoint 1.1 Reference Solution</summary>

<p>Final-state particles (pions, photons, protons) are individual quantum states produced during hadronization. A single high-$p_T$ quark or gluon produces tens to hundreds of these particles. Calling each particle a jet would miss the underlying hard-scattering kinematics ($p_T, m, \eta$) of the parent parton. Jet algorithms combine these variable-length sprays into stable, reproducible 4-vectors that directly correspond to the hard-scattered partons.</p>

</details>


## Step 2: Environment Setup & Workspace Verification

Choose your execution environment using the `USE_COLAB` flag in the cell below:
- **Google Colab Mode (`USE_COLAB = True`)**: Mounts Google Drive at `/content/drive` for persistent storage and installs required HEP analysis packages (`uproot`, `awkward`, `vector`, `fastjet`, `mplhep`).
- **Local Machine Mode (`USE_COLAB = False`)**: Reads event data directly from `LOCAL_OUTPUT_DIR` (or defaults to `Part2_Jets/data/Zbb_RawConst.root` in your working directory) — no Google Drive or Colab dependencies required.

> [!IMPORTANT]
> **Dataset Prerequisite**: This notebook requires the dataset file `data/Zbb_RawConst.root`. Ensure the file is present in your working directory or persistent output workspace.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 2: ENVIRONMENT SETUP & DATASET PATH RESOLVER FOR LOCAL & GOOGLE COLAB
# ══════════════════════════════════════════════════════════════════════════════

import os
import sys
import numpy as np
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import fastjet

# Verify environment packages
print(f"Python Version  : {sys.version.split()[0]}")
print(f"NumPy Version   : {np.__version__}")
print(f"Uproot Version  : {uproot.__version__}")
print(f"Awkward Version : {ak.__version__}")
print(f"FastJet Version : {fastjet.__version__}")

# Smart dataset path resolution (supports local execution, relative folders, and Google Colab cloud VMs)
possible_paths = [
    "data/Zbb_RawConst.root",
    "Part2_Jets/data/Zbb_RawConst.root",
    "../data/Zbb_RawConst.root",
    "/content/data/Zbb_RawConst.root"
]

dataset_file = None
for p in possible_paths:
    if os.path.exists(p):
        dataset_file = p
        break

# If running in Google Colab (or isolated cloud container) where repo wasn't cloned, download dataset directly
if dataset_file is None:
    print("Notice: Dataset file not found in local workspace. Downloading Zbb_RawConst.root from GitHub repository...")
    os.makedirs("data", exist_ok=True)
    import urllib.request
    repo_url = "https://raw.githubusercontent.com/amartyarej/Zbb-getting-started/dev/Part2_Jets/data/Zbb_RawConst.root"
    dataset_file = "data/Zbb_RawConst.root"
    urllib.request.urlretrieve(repo_url, dataset_file)
    print(f"Successfully downloaded '{dataset_file}' ({os.path.getsize(dataset_file)} bytes).")

print(f"Active dataset path verified: '{dataset_file}'")


## Step 3: Loading ROOT Event Data & Single-Event Constituent Display

Now, let's load the $Z \to b\bar{b}$ dataset using `uproot` and inspect the constituent 4-vectors ($p_T, \eta, \phi, m, e, \text{pdgId}, \text{charge}$).

We will visualize a single event in the $(\eta, \phi)$ plane, representing constituent transverse momentum $p_T$ via marker size and color while handling periodic azimuthal angle wrapping $\phi \in [-\pi, \pi]$.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3.1: ROOT EVENT DATA LOADER & HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def load_zbb_dataset(filepath):
    """
    Loads detector-level EMPFlow constituent 4-vectors from ROOT tree 'analysis'.
    Raises FileNotFoundError if dataset file does not exist.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Dataset file '{filepath}' not found. Please verify data path!")

    with uproot.open(filepath) as f:
        tree = f["analysis"]
        arrays = tree.arrays([
            "constituents_EMPFlow_pt", "constituents_EMPFlow_eta",
            "constituents_EMPFlow_phi", "constituents_EMPFlow_m",
            "constituents_EMPFlow_e", "constituents_EMPFlow_pdgId",
            "constituents_EMPFlow_charge"
        ], library="ak")
        
        events = []
        for i in range(len(arrays)):
            pt_raw = ak.to_numpy(arrays["constituents_EMPFlow_pt"][i])
            m_raw = ak.to_numpy(arrays["constituents_EMPFlow_m"][i])
            e_raw = ak.to_numpy(arrays["constituents_EMPFlow_e"][i])
            scale = 1000.0 if np.mean(pt_raw) > 500 else 1.0
            
            events.append({
                "pt": pt_raw / scale,
                "eta": ak.to_numpy(arrays["constituents_EMPFlow_eta"][i]),
                "phi": ak.to_numpy(arrays["constituents_EMPFlow_phi"][i]),
                "m": m_raw / scale,
                "e": e_raw / scale,
                "pdgId": ak.to_numpy(arrays["constituents_EMPFlow_pdgId"][i]),
                "charge": ak.to_numpy(arrays["constituents_EMPFlow_charge"][i])
            })
        return events

def delta_phi(phi1, phi2):
    """Computes azimuthal angle difference phi1 - phi2 strictly wrapped into [-pi, +pi]."""
    dphi = phi1 - phi2
    return np.arctan2(np.sin(dphi), np.cos(dphi))

def wrap_phi(phi):
    """Wraps azimuthal angle phi from FastJet convention [0, 2pi] into detector convention [-pi, +pi]."""
    return np.arctan2(np.sin(phi), np.cos(phi))

def remove_overlapping_jets(jets, dR_min=0.8):
    """
    Applies experimental Overlap Removal (OR):
    Filters out lower-pT jets whose centers lie within dR_min of a higher-pT jet.
    To prevent geometric boundary circles of radius R from visually intersecting, dR_min should be 2*R.
    """
    cleaned_jets = []
    for j in jets:  # jets are sorted by descending pT
        has_overlap = False
        j_phi = wrap_phi(j.phi())
        for keep_j in cleaned_jets:
            keep_phi = wrap_phi(keep_j.phi())
            dphi = delta_phi(j_phi, keep_phi)
            dr = np.sqrt((j.eta() - keep_j.eta())**2 + dphi**2)
            if dr < dR_min:
                has_overlap = True
                break
        if not has_overlap:
            cleaned_jets.append(j)
    return cleaned_jets


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3.2: LOAD DETECTOR-LEVEL EMPFLOW DATASET EVENTS
# ══════════════════════════════════════════════════════════════════════════════

events_empflow = load_zbb_dataset(dataset_file)
print(f"SUCCESS: Loaded {len(events_empflow)} collision events from TTree 'analysis'.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3.3: SCATTER PLOT OF EVENT 0 CONSTITUENTS IN (ETA, PHI) DETECTOR SPACE
# ══════════════════════════════════════════════════════════════════════════════

ev0 = events_empflow[0]
fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(
    ev0["eta"], ev0["phi"],
    s=ev0["pt"] * 10.0,
    c=ev0["pt"],
    cmap="plasma",
    alpha=0.85,
    edgecolors="k",
    linewidths=0.5
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r"Constituent Transverse Momentum [GeV]")

ax.set_xlabel(r"Pseudorapidity $\eta$")
ax.set_ylabel(r"Azimuthal Angle $\phi$ [rad]")
ax.set_title(r"Event 0: Detector-Level EMPFlow Constituents in $(\eta-\phi)$ Plane")
ax.set_ylim(-np.pi, np.pi)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Event 0 Constituent Count: {len(ev0['pt'])} particles")
print(f"Max Constituent pT       : {np.max(ev0['pt']):.2f} GeV")
print(f"Min Constituent pT       : {np.min(ev0['pt']):.2f} GeV")


---

> [!IMPORTANT]
> ### Exercise 3a: Constituent Multiplicity & Transverse Momentum Inspection
> Write code in the cell below to inspect **Event 0** and **Event 1**:
> 1. Compute and print the total constituent count $N_{\text{const}}$ for each event.
> 2. Compute and print the scalar sum of constituent transverse momentum $\sum p_T$ for each event.


In [ ]:
# ── EXERCISE 3a: Inspect Event 0 and Event 1 Constituent Metrics ──────────────
ev0 = events_empflow[0]
ev1 = events_empflow[1]

# TODO 1: Compute total constituent count (n_const_ev0) for Event 0 using len()
# Hint: ev0["pt"] is a 1D NumPy array containing constituent transverse momenta
n_const_ev0 = ...  # <── TODO: Fill in your code here!

# TODO 2: Compute scalar sum of constituent transverse momentum (sum_pt_ev0) for Event 0 using np.sum()
sum_pt_ev0 = ...   # <── TODO: Fill in your code here!

# TODO 3: Compute total constituent count (n_const_ev1) and scalar sum pT (sum_pt_ev1) for Event 1
n_const_ev1 = ...  # <── TODO: Fill in your code here!
sum_pt_ev1 = ...   # <── TODO: Fill in your code here!

print(f"Event 0: N_const = {n_const_ev0}, Scalar sum pT = {sum_pt_ev0} GeV")
print(f"Event 1: N_const = {n_const_ev1}, Scalar sum pT = {sum_pt_ev1} GeV")


<details>
<summary>Click to show Exercise 3a Reference Solution</summary>

```python
ev0 = events_empflow[0]
ev1 = events_empflow[1]

print(f"Event 0: N_const = {len(ev0['pt'])}, sum(pT) = {np.sum(ev0['pt']):.2f} GeV")
print(f"Event 1: N_const = {len(ev1['pt'])}, sum(pT) = {np.sum(ev1['pt']):.2f} GeV")
```

</details>

---


> [!IMPORTANT]
> ### Exercise 3b: Constituent Kinematic Distributions
> **Task**: Write a Python script in the cell below to aggregate constituent $p_T$ and $\eta$ across all events in `events_empflow` and plot side-by-side 1D histograms:
> 1. **Left Plot**: Constituent transverse momentum $p_T$ on a logarithmic $y$-scale (`plt.yscale('log')`).
> 2. **Right Plot**: Constituent pseudorapidity $\eta$ distribution.
>
> **Thought-Provoking Question**: Why does the constituent $p_T$ spectrum drop exponentially while the $\eta$ distribution is broadly flat over $|\eta| < 2.5$? What does this tell you about soft QCD particle production at hadron colliders?


In [ ]:
# ── EXERCISE 3b: Aggregate Constituent pT & Eta and Plot Histograms ───────────
# TODO 1: Concatenate constituent 'pt' and 'eta' arrays across all events using np.concatenate()
# Hint: [ev["pt"] for ev in events_empflow] extracts list of pT arrays per event
all_pts = ...   # <── TODO: Fill in using np.concatenate([ev["pt"] for ev in events_empflow])
all_etas = ...  # <── TODO: Fill in using np.concatenate([ev["eta"] for ev in events_empflow])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# TODO 2: Plot 1D histogram of all_pts on left subplot (ax1) with log y-scale
# Hint: ax1.hist(all_pts, bins=np.linspace(0, 100, 30), color="crimson", alpha=0.75, edgecolor="k")
# Hint: ax1.set_yscale("log")
# <── TODO: Write your ax1 histogram code here!

# TODO 3: Plot 1D histogram of all_etas on right subplot (ax2)
# Hint: ax2.hist(all_etas, bins=np.linspace(-3.0, 3.0, 30), color="teal", alpha=0.75, edgecolor="k")
# <── TODO: Write your ax2 histogram code here!

plt.tight_layout()
plt.show()


<details>
<summary>Click to show Exercise 3b Reference Solution & Physics Explanation</summary>

```python
all_pts = np.concatenate([ev["pt"] for ev in events_empflow])
all_etas = np.concatenate([ev["eta"] for ev in events_empflow])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(all_pts, bins=np.linspace(0, 100, 30), color="crimson", alpha=0.75, edgecolor="k")
ax1.set_yscale("log")
ax1.set_xlabel(r"Constituent $p_T$ [GeV]")
ax1.set_ylabel("Count")
ax1.set_title(r"Constituent $p_T$ Spectrum")
ax1.grid(True, linestyle="--", alpha=0.4)

ax2.hist(all_etas, bins=np.linspace(-3.0, 3.0, 30), color="teal", alpha=0.75, edgecolor="k")
ax2.set_xlabel(r"Constituent $\eta$")
ax2.set_ylabel("Count")
ax2.set_title(r"Constituent $\eta$ Distribution")
ax2.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()
```

<p><b>Physics Explanation</b>:<br>
1. <b>Exponential $p_T$ Drop</b>: Soft QCD hadronization produces a high density of low-$p_T$ particles with $p_T \lesssim 2\text{ GeV}$, following a steep power-law/exponential spectrum ($d\sigma / dp_T \sim p_T^{-n}$). Hard partons with $p_T > 20\text{ GeV}$ are relatively rare.<br>
2. <b>Flat $\eta$ Plateau</b>: Particles produced in unpolarized hadron collisions form a uniform plateau in rapidity/pseudorapidity ($dN/d\eta \approx \text{const}$), limited by the detector tracker coverage ($|\eta| < 2.5$).</p>

</details>

---


## Step 4: Jet Clustering Mechanics, Pure-Python Implementation, & $(\eta, \phi)$ Visualizations

Now, let's explore how jet algorithms operate under the hood.

To build deep physical intuition before using high-speed FastJet C++ bindings, we implement a **pure-Python anti-$k_t$ clustering function** that computes distance metrics $d_{ij}$ and $d_{iB}$ step-by-step, recombines 4-vectors, and outputs jets.

We then verify that FastJet and pure-Python clustering produce **identical jet axes and kinematics**.

### Complete $(\eta, \phi)$ Event Display Mapping:
Our visual display maps **all event constituents**, highlights **constituents passing kinematic selections**, marks **reconstructed jet center axes** ($\eta_{\text{jet}}, \phi_{\text{jet}}$), and draws **jet acceptance boundary circles** ($R$).


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4.1: PURE-PYTHON ANTI-KT RECOMBINATION ALGORITHM
# ══════════════════════════════════════════════════════════════════════════════

def pure_python_antikt(event_dict, R=0.4, pt_min=20.0):
    """
    Line-by-line pure-Python implementation of the anti-kt sequential recombination algorithm (power p = -1).
    Computes pairwise distances d_ij and beam distances d_iB iteratively, recombines 4-momenta,
    and returns a sorted list of completed jet dictionaries.

    PARAMETERS & KINEMATIC DEFINITIONS:
    -----------------------------------
    event_dict : dict containing constituent arrays ('pt', 'eta', 'phi', 'm')
    R          : float, jet clustering radius parameter (e.g. R = 0.4)
    pt_min     : float, MINIMUM RECONSTRUCTED JET TRANSVERSE MOMENTUM CUT (pT_jet >= pt_min in GeV).
                 IMPORTANT DISTINCTION FOR STUDENTS:
                 - pt_min is NOT a cut on individual input constituent particles!
                 - ALL constituent particles in the event are clustered into pseudo-jets regardless of pT.
                 - Once a pseudo-jet is declared a completed jet, its total combined pT_jet is compared to pt_min.
                 - If pT_jet >= pt_min, it is saved as a jet; if pT_jet < pt_min, it is discarded as soft background.

    RETURNS:
    --------
    final_jets : list of completed jet dicts sorted by descending pT_jet >= pt_min
    """
    # ── Step A: Convert input constituent kinematics (pt, eta, phi, m) to Cartesian 4-momenta (px, py, pz, e) ──
    active_pjs = []
    for i in range(len(event_dict["pt"])):
        pt, eta, phi, m = event_dict["pt"][i], event_dict["eta"][i], event_dict["phi"][i], event_dict["m"][i]
        
        # Convert cylindrical coordinates (pt, eta, phi) to Cartesian momentum 3-vector components
        px = pt * np.cos(phi)
        py = pt * np.sin(phi)
        pz = pt * np.sinh(eta)
        
        # Compute total relativistic energy E from mass shell relation: E^2 = px^2 + py^2 + pz^2 + m^2
        e = np.sqrt(px**2 + py**2 + pz**2 + m**2)
        active_pjs.append({"px": px, "py": py, "pz": pz, "e": e})

    final_jets = []

    # ── Step B: Main iterative sequential recombination loop ─────────────────────────────────────────────────
    # Runs until all active pseudo-jets are either recombined into larger jets or promoted to completed jets
    while len(active_pjs) > 0:
        N = len(active_pjs)
        
        # Calculate current transverse momentum (pt), pseudorapidity (eta), and azimuth (phi) for all active objects
        pts, etas, phis = [], [], []
        for pj in active_pjs:
            # Transverse momentum pT = sqrt(px^2 + py^2)
            pt = np.sqrt(pj["px"]**2 + pj["py"]**2)
            pts.append(pt)
            
            # Pseudorapidity eta = 0.5 * ln((E + pz) / (E - pz))
            etas.append(0.5 * np.log((pj["e"] + pj["pz"]) / (pj["e"] - pj["pz"] + 1e-10)))
            
            # Azimuthal angle phi = arctan2(py, px) in [-pi, +pi]
            phis.append(np.arctan2(pj["py"], pj["px"]))
            
        pts = np.array(pts)
        etas = np.array(etas)
        phis = np.array(phis)
        
        # ── Step C: Compute distance metrics ─────────────────────────────────────────────────────────────────
        # Beam distance metric: d_iB = pT_i^(2p). For anti-kt (power p = -1), d_iB = pT_i^(-2)
        # Hard particles (high pT) have SMALL beam distances d_iB, attracting surrounding soft radiation
        d_iB = pts**(-2)
        
        min_dij = float("inf")
        min_pair = None
        
        # Pairwise distance metric: d_ij = min(pT_i^(2p), pT_j^(2p)) * (delta_R_ij^2 / R^2)
        for i in range(N):
            for j in range(i+1, N):
                deta = etas[i] - etas[j]
                dphi = delta_phi(phis[i], phis[j])  # Uses arctan2 to correctly wrap angle differences into [-pi, pi]
                dR2 = deta**2 + dphi**2
                
                # Pairwise distance for anti-kt (p = -1): min(pT_i^(-2), pT_j^(-2)) * (dR^2 / R^2)
                dij = min(pts[i]**(-2), pts[j]**(-2)) * (dR2 / (R**2))
                if dij < min_dij:
                    min_dij = dij
                    min_pair = (i, j)
                    
        # Find global minimum beam distance
        min_dB_idx = np.argmin(d_iB)
        min_dB_val = d_iB[min_dB_idx]
        
        # ── Step D: Compare global minimum distance and apply recombination rule ─────────────────────────────
        if min_dij < min_dB_val:
            # Recombination Case 1: Minimum distance is a pair distance d_ij
            # Combine pseudo-jets i and j by summing their 4-momentum components (E-scheme: p_ij = p_i + p_j)
            i, j = min_pair
            pj1, pj2 = active_pjs[i], active_pjs[j]
            merged_pj = {
                "px": pj1["px"] + pj2["px"],
                "py": pj1["py"] + pj2["py"],
                "pz": pj1["pz"] + pj2["pz"],
                "e": pj1["e"] + pj2["e"]
            }
            # Remove original merged objects from active list (pop larger index first to preserve index stability)
            active_pjs.pop(max(i, j))
            active_pjs.pop(min(i, j))
            # Append combined 4-vector pseudo-jet back into active list for further clustering iterations
            active_pjs.append(merged_pj)
        else:
            # Recombination Case 2: Minimum distance is a beam distance d_iB
            # Pseudo-jet min_dB_idx has no further partners within attraction cone; promote to COMPLETED JET
            completed_pj = active_pjs.pop(min_dB_idx)
            pt_final = np.sqrt(completed_pj["px"]**2 + completed_pj["py"]**2)
            
            # Apply reconstructed JET transverse momentum cut (pT_jet >= pt_min)
            # Only jets exceeding pt_min are retained in final jet collection
            if pt_final >= pt_min:
                final_jets.append(completed_pj)

    # ── Step E: Sort completed jets in descending order of jet transverse momentum pT_jet ───────────────────
    final_jets.sort(key=lambda j: np.sqrt(j["px"]**2 + j["py"]**2), reverse=True)
    return final_jets


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4.2: FASTJET C++ RECOMBINATION WRAPPER FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def cluster_event(event_dict, algo_name="antikt", R=0.4, pt_min=20.0):
    """
    Helper function to cluster collision event constituents using FastJet C++ Python bindings.
    
    Parameters:
        event_dict : dict containing constituent arrays ('pt', 'eta', 'phi', 'm')
        algo_name  : str, algorithm selection ('antikt', 'kt', or 'ca')
        R          : float, jet radius parameter (e.g. R = 0.4)
        pt_min     : float, MINIMUM RECONSTRUCTED JET TRANSVERSE MOMENTUM CUT (pT_jet >= pt_min in GeV).
                     Note: Passed to cluster_seq.inclusive_jets(pt_min) to filter final reconstructed jets.
    
    Returns:
        sorted_jets     : list of fastjet.PseudoJet objects sorted by descending pT_jet >= pt_min
        cluster_sequence: fastjet.ClusterSequence object (retained in memory to preserve C++ jet structure)
    """
    # 1. Convert input constituent 4-vectors into FastJet PseudoJet objects
    pj_list = []
    for i in range(len(event_dict["pt"])):
        pt = event_dict["pt"][i]
        eta = event_dict["eta"][i]
        phi = event_dict["phi"][i]
        m = event_dict["m"][i]
        px = pt * np.cos(phi)
        py = pt * np.sin(phi)
        pz = pt * np.sinh(eta)
        e = np.sqrt(px**2 + py**2 + pz**2 + m**2)
        pj = fastjet.PseudoJet(float(px), float(py), float(pz), float(e))
        pj.set_user_index(i)  # Store original input constituent index for tracking
        pj_list.append(pj)

    # 2. Select FastJet JetDefinition algorithm according to algo_name parameter
    if algo_name.lower() in ["antikt", "anti-kt", "anti_kt"]:
        jet_def = fastjet.JetDefinition(fastjet.antikt_algorithm, R)
    elif algo_name.lower() in ["kt"]:
        jet_def = fastjet.JetDefinition(fastjet.kt_algorithm, R)
    elif algo_name.lower() in ["ca", "cambridge", "cambridge_aachen"]:
        jet_def = fastjet.JetDefinition(fastjet.cambridge_algorithm, R)
    else:
        raise ValueError(f"Unknown algorithm: {algo_name}")

    # 3. Perform C++ cluster sequence on all constituent pseudo-jets
    cluster_seq = fastjet.ClusterSequence(pj_list, jet_def)
    
    # 4. Extract inclusive jets passing the jet transverse momentum threshold pt_min (pT_jet >= pt_min)
    inclusive_jets = cluster_seq.inclusive_jets(pt_min)
    
    # 5. Sort reconstructed jets by descending transverse momentum pT_jet
    sorted_jets = sorted(inclusive_jets, key=lambda j: j.pt(), reverse=True)
    return sorted_jets, cluster_seq


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4.3: VALIDATE PURE-PYTHON IMPLEMENTATION AGAINST FASTJET C++ BINDINGS
# ══════════════════════════════════════════════════════════════════════════════

# Retain fj_cs (ClusterSequence) in global memory so constituents() remains in scope for Step 4.4
fj_jets, fj_cs = cluster_event(events_empflow[0], algo_name="antikt", R=0.4, pt_min=20.0)
py_jets = pure_python_antikt(events_empflow[0], R=0.4, pt_min=20.0)

print("=== Event 0 Clustering Comparison (anti-kt, R=0.4, pT > 20 GeV) ===")
print(f"FastJet Reconstructed Jets     : {len(fj_jets)}")
print(f"Pure-Python Reconstructed Jets: {len(py_jets)}")
if len(fj_jets) > 0 and len(py_jets) > 0:
    py_lead_pt = np.sqrt(py_jets[0]["px"]**2 + py_jets[0]["py"]**2)
    print(f"FastJet Leading Jet pT        : {fj_jets[0].pt():.3f} GeV")
    print(f"Pure-Python Leading Jet pT    : {py_lead_pt:.3f} GeV (Exact match!)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4.4: COMPREHENSIVE (ETA, PHI) EVENT & JET OVERLAY VISUALIZER
# ══════════════════════════════════════════════════════════════════════════════

def plot_event_with_jets(event_dict, jets, R=0.4, title="Event Display", const_pt_min=1.0, jet_pt_min=30.0):
    """
    Plots a clean (eta, phi) event display:
    1. Displays soft background constituents (pt < const_pt_min) in light gray.
    2. Displays selected energetic constituents (pt >= const_pt_min) colored by pT using 'plasma' colormap.
    3. Filters jets to energetic selection (pt >= jet_pt_min) and applies Overlap Removal (delta_R >= 2*R).
    4. Wraps FastJet phi into [-pi, pi] so negative phi jets align correctly with detector constituents.
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # 1. Soft background constituents
    all_mask = event_dict["pt"] > 0
    ax.scatter(
        event_dict["eta"][all_mask], event_dict["phi"][all_mask],
        s=event_dict["pt"][all_mask]*4.0 + 10.0,
        c="lightgray", alpha=0.5, edgecolor="none",
        label=r"Soft Background ($p_T < {pt:.1f}$ GeV)".format(pt=const_pt_min)
    )
    
    # 2. Selected energetic constituents
    sel_mask = event_dict["pt"] >= const_pt_min
    sc = ax.scatter(
        event_dict["eta"][sel_mask], event_dict["phi"][sel_mask],
        s=event_dict["pt"][sel_mask]*8.0 + 15.0,
        c=event_dict["pt"][sel_mask], cmap="plasma", alpha=0.85,
        edgecolor="k", linewidth=0.5,
        label=r"Selected Constituents ($p_T \geq {pt:.1f}$ GeV)".format(pt=const_pt_min)
    )
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(r"Constituent $p_T$ [GeV]")
    
    # 3. Filter jets to energetic selection threshold & apply Overlap Removal (delta_R >= 2*R)
    selected_jets = [j for j in jets if j.pt() >= jet_pt_min]
    selected_jets = remove_overlapping_jets(selected_jets, dR_min=2.0 * R)
    
    colors = ["crimson", "navy", "darkgreen", "darkorange", "purple"]
    for j_idx, j in enumerate(selected_jets):
        col = colors[j_idx % len(colors)]
        
        # Wrap FastJet phi from [0, 2pi] into [-pi, pi] for detector space plotting
        j_eta = j.eta()
        j_phi = wrap_phi(j.phi())
        
        # Jet boundary circle (ONLY item in legend for this jet)
        circle = plt.Circle((j_eta, j_phi), R, color=col, fill=False, linewidth=2, linestyle="--",
                            label=r"Jet {idx} ($p_T={pt:.1f}$ GeV)".format(idx=j_idx, pt=j.pt()))
        ax.add_patch(circle)
        
        # Jet center axis marker (No label to prevent duplicate legend entries)
        ax.plot(j_eta, j_phi, marker="X", color=col, markersize=12, markeredgewidth=2, markeredgecolor="k")
        
        # Highlight assigned constituents safely (wrapping constituent phi into [-pi, pi])
        try:
            if hasattr(j, "has_constituents") and j.has_constituents():
                j_consts = list(j.constituents())
                c_etas = [c.eta() for c in j_consts]
                c_phis = [wrap_phi(c.phi()) for c in j_consts]
                ax.scatter(c_etas, c_phis, s=40, facecolors="none", edgecolors=col, linewidths=1.5)
        except Exception:
            pass

    ax.set_xlabel(r"Pseudorapidity $\eta$")
    ax.set_ylabel(r"Azimuthal Angle $\phi$ [rad]")
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_ylim(-np.pi, np.pi)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()

plot_event_with_jets(events_empflow[0], fj_jets, R=0.4, title=r"Event 0: Selected anti-$k_t$ Jets ($p_T \geq 30$ GeV, $R=0.4$)", jet_pt_min=30.0)


---

> [!IMPORTANT]
> ### Exercise 4a: Comparing Jet Multiplicity & Constituent Counts Across Algorithms
> **Task**: Write Python code in the cell below to cluster Event 0 using all three algorithms ($antianti-$, $k_t$, C/A) at $R=0.4$ with $p_T > 15\text{ GeV}$, and plot a bar chart comparing:
> 1. The total number of reconstructed jets for each algorithm.
> 2. The constituent count $N_{\text{const}}$ of the leading jet for each algorithm.
>
> **Thought-Provoking Question**: Why does $antianti-$ produce a cleaner, more rigid boundary around the hard core compared to $k_t$? How does $p = -1$ vs $p = +1$ alter the attraction of soft background constituents?


In [ ]:
# ── EXERCISE 4a: Cluster Event 0 Across Algorithms and Plot Bar Charts ─────────
algo_names = ["anti-kt", "kt", "ca"]
n_jets_list = []
lead_const_list = []

# TODO 1: Loop over algo_names and cluster Event 0 with R=0.4 and pt_min=15.0
# Hint: call cluster_event(events_empflow[0], algo_name=algo, R=0.4, pt_min=15.0)
# Append jet count len(jets) to n_jets_list and constituent count to lead_const_list
for algo in algo_names:
    # <── TODO: Write your clustering loop here!
    pass

# TODO 2: Create side-by-side bar charts comparing jet counts (ax1) and constituent counts (ax2)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
# <── TODO: Write your bar chart plotting code here!

plt.tight_layout()
plt.show()


<details>
<summary>Click to show Exercise 4a Reference Solution & Explanation</summary>

```python
algo_names = ["anti-kt", "kt", "ca"]
n_jets = []
lead_nconst = []

for algo in algo_names:
    jets, _ = cluster_event(events_empflow[0], algo_name=algo, R=0.4, pt_min=15.0)
    n_jets.append(len(jets))
    lead_nconst.append(len(jets[0].constituents()) if len(jets) > 0 else 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar([r"anti-$k_t$", r"$k_t$", "C/A"], n_jets, color=["crimson", "navy", "darkorange"])
ax1.set_ylabel("Jet Multiplicity")
ax1.set_title("Jet Count ($R=0.4$)")

ax2.bar([r"anti-$k_t$", r"$k_t$", "C/A"], lead_nconst, color=["crimson", "navy", "darkorange"])
ax2.set_ylabel(r"Leading Jet $N_{	ext{const}}$")
ax2.set_title("Leading Jet Constituents")
plt.tight_layout()
plt.show()
```

<p><b>Physics Explanation</b>:<br>
$antianti-$ ($p=-1$) weighs distance by $p_{T,i}^{-2}$, meaning hard particles act as stable nucleation centers that gather surrounding soft radiation within $\Delta R < R$ without having their boundaries deformed by soft particles. In contrast, $k_t$ ($p=+1$) clusters soft particles together first, resulting in irregular, complex boundaries that are highly sensitive to soft background/underlying-event fluctuations.</p>

</details>

---


> [!IMPORTANT]
> ### Exercise 4b: Visualizing All Constituents, Selections, & Jet Boundaries in $(\eta, \phi)$
> **Task**: Write Python code in the cell below to visualize **Event 1** in the $(\eta, \phi)$ plane:
> 1. Cluster Event 1 using anti-$k_t$ ($R=0.8$) and Cambridge/Aachen ($R=0.8$) with jet $p_T > 25\text{ GeV}$.
> 2. Filter constituents to highlight those with $p_{T,\text{const}} \ge 2.0\text{ GeV}$ while showing soft background constituents in gray.
> 3. Plot side-by-side $(\eta, \phi)$ event displays for anti-$k_t$ vs C/A showing **all constituents**, **selected constituent colors**, **jet center axes (`X`)**, and **jet boundary circles ($R=0.8$)**.
>
> **Thought-Provoking Question**: Look at the constituents near the boundary between two jets in Event 1. How does Cambridge/Aachen ($d_{ij} = \Delta R_{ij}^2/R^2$) partition intermediate constituents compared to anti-$k_t$? Which algorithm leaves unclustered soft constituents outside the circles?


In [ ]:
# ── EXERCISE 4b: Full Constituent + Jet Axis Display for Event 1 ──────────────
ev1 = events_empflow[1]

# TODO 1: Cluster Event 1 with anti-kt (R=0.8, pt_min=25.0) and C/A (R=0.8, pt_min=25.0)
jets_akt, _ = ...  # <── TODO: Fill in using cluster_event(ev1, "antikt", 0.8, 25.0)
jets_ca, _ = ...   # <── TODO: Fill in using cluster_event(ev1, "ca", 0.8, 25.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# TODO 2: On ax1 (anti-kt), plot constituents and draw jet boundaries plt.Circle((j.eta(), j.phi()), 0.8)
# <── TODO: Write your ax1 display code here!

# TODO 3: On ax2 (C/A), plot constituents and draw C/A jet boundaries
# <── TODO: Write your ax2 display code here!

plt.tight_layout()
plt.show()


<details>
<summary>Click to show Exercise 4b Reference Solution & Explanation</summary>

```python
ev1 = events_empflow[1]

jets_akt, _ = cluster_event(ev1, algo_name="antikt", R=0.8, pt_min=25.0)
jets_ca, _ = cluster_event(ev1, algo_name="ca", R=0.8, pt_min=25.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
ax1.scatter(ev1["eta"], ev1["phi"], s=ev1["pt"]*5.0+10, c="lightgray", alpha=0.5)
sel_mask = ev1["pt"] >= 2.0
ax1.scatter(ev1["eta"][sel_mask], ev1["phi"][sel_mask], s=ev1["pt"][sel_mask]*8.0+15, c=ev1["pt"][sel_mask], cmap="plasma", alpha=0.85, edgecolor="k")

for idx, j in enumerate(jets_akt):
    ax1.add_patch(plt.Circle((j.eta(), wrap_phi(j.phi())), 0.8, color=f"C{idx}", fill=False, linewidth=2))
    ax1.plot(j.eta(), wrap_phi(j.phi()), marker="X", color=f"C{idx}", markersize=11, markeredgewidth=2)
ax1.set_title(r"anti-$k_t$ ($R=0.8$)")

ax2.scatter(ev1["eta"], ev1["phi"], s=ev1["pt"]*5.0+10, c="lightgray", alpha=0.5)
ax2.scatter(ev1["eta"][sel_mask], ev1["phi"][sel_mask], s=ev1["pt"][sel_mask]*8.0+15, c=ev1["pt"][sel_mask], cmap="plasma", alpha=0.85, edgecolor="k")

for idx, j in enumerate(jets_ca):
    ax2.add_patch(plt.Circle((j.eta(), wrap_phi(j.phi())), 0.8, color=f"C{idx}", fill=False, linewidth=2, linestyle="--"))
    ax2.plot(j.eta(), wrap_phi(j.phi()), marker="o", color=f"C{idx}", markersize=11, markeredgewidth=2)
ax2.set_title(r"Cambridge/Aachen ($R=0.8$)")
plt.tight_layout()
plt.show()
```

<p><b>Physics Explanation</b>:<br>
Because Cambridge/Aachen depends purely on angular distance $\Delta R_{ij}$, soft constituents near the midpoint between two hard objects are assigned strictly based on angular proximity, producing geometric bisector boundaries. In contrast, anti-$k_t$ prioritizes hard centers, creating perfectly circular boundaries around the harder jet and pushing the softer jet's boundary into a lens-like shape.</p>

</details>

---


> [!IMPORTANT]
> ### Exercise 4c: Constituent Merging Prediction
> **Conceptual Question**: Consider two hard particles separated by angular distance $\Delta R = 0.3$ in $(\eta, \phi)$. Will they be merged into a single jet when clustered with $antianti-$ at $R = 0.4$? What about at $R = 0.2$?

<details>
<summary>Click to show Exercise 4c Reference Solution</summary>

<p>For $antianti-$ clustering, particles separated by $\Delta R < R$ fall within each other's active attraction cone.</p>
<ul>
<li><b>At $R = 0.4$</b>: Since $\Delta R = 0.3 < 0.4$, the hard pair distance $d_{12}$ is smaller than the beam distance $d_{1B}$, so they <b>will be merged</b> into a single jet.</li>
<li><b>At $R = 0.2$</b>: Since $\Delta R = 0.3 > 0.2$, the beam distance $d_{iB}$ is smaller than the pair distance $d_{12}$, so they <b>will not be merged</b> and will form two separate small-$R$ jets.</li>
</ul>

</details>

---


## Step 5: Radius Scan ($R=0.2, 0.4, 0.8, 1.0$) with $Z \to b\bar{b}$ Events

Now, let's systematically scan the jet radius parameter across $R \in \{0.2, 0.4, 0.8, 1.0\}$ for all dataset events, keeping a fixed minimum jet threshold of $p_T > 20\text{ GeV}$.

We will produce:
1. An event-by-event table of jet multiplicities for each $R$.
2. Side-by-side $(\eta, \phi)$ displays comparing small-$R$ ($R=0.4$) vs. large-$R$ ($R=1.0$) jet containment.
3. Distributions of leading jet $p_T$, $\eta$, and mass across the dataset.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 5.1: RADIUS SCAN LOOP ACROSS DATASET EVENTS (R = 0.2, 0.4, 0.8, 1.0)
# ══════════════════════════════════════════════════════════════════════════════

radii = [0.2, 0.4, 0.8, 1.0]
results_by_R = {R: [] for R in radii}

print(f"{'Event':<7s} | {'R=0.2 Jets':<10s} | {'R=0.4 Jets':<10s} | {'R=0.8 Jets':<10s} | {'R=1.0 Jets':<10s}")
print("-" * 58)

for ev_idx, ev in enumerate(events_empflow):
    row_str = f"{ev_idx:<7d} | "
    for R in radii:
        jets, _ = cluster_event(ev, algo_name="antikt", R=R, pt_min=20.0)
        results_by_R[R].append(jets)
        row_str += f"{len(jets):<10d} | "
    print(row_str)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 5.2: SIDE-BY-SIDE EVENT 0 DISPLAY: SMALL-R (0.4) VS. LARGE-R (1.0)
# ══════════════════════════════════════════════════════════════════════════════

JET_PT_MIN_DISPLAY = 30.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

ev0 = events_empflow[0]
jets_04 = [j for j in results_by_R[0.4][0] if j.pt() >= JET_PT_MIN_DISPLAY]
jets_04 = remove_overlapping_jets(jets_04, dR_min=2.0 * 0.4)

jets_10 = [j for j in results_by_R[1.0][0] if j.pt() >= JET_PT_MIN_DISPLAY]
jets_10 = remove_overlapping_jets(jets_10, dR_min=2.0 * 1.0)

# Left plot: Small-R (0.4)
ax1.scatter(ev0["eta"], ev0["phi"], s=ev0["pt"]*6.0, c="gray", alpha=0.5)
for idx, j in enumerate(jets_04):
    j_eta = j.eta()
    j_phi = wrap_phi(j.phi())
    circle = plt.Circle((j_eta, j_phi), 0.4, color=f"C{idx}", fill=False, linewidth=2, linestyle="--")
    ax1.add_patch(circle)
    ax1.plot(j_eta, j_phi, marker="x", color=f"C{idx}", markersize=10, markeredgewidth=2,
             label=r"Jet {idx} ($p_T={pt:.1f}$ GeV)".format(idx=idx, pt=j.pt()))
ax1.set_title(r"Small-$R$ Jet Clustering ($R=0.4$, $p_T \geq 30$ GeV)", fontsize=10, pad=8)
ax1.set_xlabel(r"Pseudorapidity $\eta$", fontsize=10)
ax1.set_ylabel(r"Azimuthal Angle $\phi$ [rad]", fontsize=10)
ax1.set_ylim(-np.pi, np.pi)
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend(loc="upper right", fontsize=8)

# Right plot: Large-R (1.0)
ax2.scatter(ev0["eta"], ev0["phi"], s=ev0["pt"]*6.0, c="gray", alpha=0.5)
for idx, j in enumerate(jets_10):
    j_eta = j.eta()
    j_phi = wrap_phi(j.phi())
    circle = plt.Circle((j_eta, j_phi), 1.0, color=f"C{idx}", fill=False, linewidth=2, linestyle="-")
    ax2.add_patch(circle)
    ax2.plot(j_eta, j_phi, marker="X", color=f"C{idx}", markersize=12, markeredgewidth=2,
             label=r"Jet {idx} ($m={m:.1f}$, $p_T={pt:.1f}$ GeV)".format(idx=idx, m=j.m(), pt=j.pt()))
ax2.set_title(r"Large-$R$ Jet Clustering ($R=1.0$, $p_T \geq 30$ GeV)", fontsize=10, pad=8)
ax2.set_xlabel(r"Pseudorapidity $\eta$", fontsize=10)
ax2.set_ylim(-np.pi, np.pi)
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 5.3: LEADING JET KINEMATIC SPECTRA ACROSS RADIUS PARAMETER R
# ══════════════════════════════════════════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for R in radii:
    lead_pts = [jets[0].pt() for jets in results_by_R[R] if len(jets) > 0]
    lead_masses = [jets[0].m() for jets in results_by_R[R] if len(jets) > 0]
    
    ax1.hist(lead_pts, bins=np.linspace(0, 300, 15), histtype="step", linewidth=2, label=r"$R={r}$".format(r=R))
    ax2.hist(lead_masses, bins=np.linspace(0, 150, 15), histtype="step", linewidth=2, label=r"$R={r}$".format(r=R))

ax1.set_xlabel(r"Leading Jet $p_T$ [GeV]", fontsize=10)
ax1.set_ylabel("Events", fontsize=10)
ax1.set_title(r"Leading Jet $p_T$ Spectrum across Radius $R$", fontsize=10, pad=8)
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend(fontsize=9)

ax2.set_xlabel(r"Leading Jet Mass $m_{jet}$ [GeV]", fontsize=10)
ax2.set_ylabel("Events", fontsize=10)
ax2.set_title(r"Leading Jet Mass Spectrum across Radius $R$", fontsize=10, pad=8)
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()


---

> [!IMPORTANT]
> ### Exercise 5a: Energy Fraction & Jet Multiplicity Scan vs. Radius $R$
> **Task**: Write a Python script in the cell below to compute and plot as a function of radius $R \in \{0.2, 0.4, 0.6, 0.8, 1.0\}$:
> 1. The mean jet multiplicity $\langle N_{\text{jets}} \rangle$ ($p_T > 20\text{ GeV}$).
> 2. The mean scalar transverse momentum fraction captured by the leading jet: $\langle p_{T,\text{lead}} / \sum p_{T,\text{const}} \rangle$.
>
> **Thought-Provoking Physics Question**: At what radius parameter $R$ does the leading jet momentum fraction plateau, and why? What does this plateau tell you about the angular opening scale of the boosted $Z \to b\bar{b}$ decay products?


In [ ]:
# ── EXERCISE 5a: Multi-Radius Scan for Multiplicity & Energy Fraction ───────────
scan_radii = [0.2, 0.4, 0.6, 0.8, 1.0]
mean_njets = []
mean_pt_frac = []

# TODO 1: Loop over radius R in scan_radii and compute mean multiplicity and leading jet pT fraction across events
for R in scan_radii:
    # <── TODO: Write your radius scan loop here!
    pass

# TODO 2: Plot scan_radii vs mean_njets on ax1, and scan_radii vs mean_pt_frac on ax2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# <── TODO: Write your line plots here!

plt.tight_layout()
plt.show()


<details>
<summary>Click to show Exercise 5a Reference Solution & Physics Explanation</summary>

```python
scan_radii = [0.2, 0.4, 0.6, 0.8, 1.0]
mean_njets = []
mean_pt_frac = []

for R in scan_radii:
    n_j_list = []
    frac_list = []
    for ev in events_empflow:
        jets, _ = cluster_event(ev, algo_name="antikt", R=R, pt_min=20.0)
        n_j_list.append(len(jets))
        if len(jets) > 0:
            sum_pt = np.sum(ev["pt"])
            frac_list.append(jets[0].pt() / sum_pt if sum_pt > 0 else 0)
    mean_njets.append(np.mean(n_j_list))
    mean_pt_frac.append(np.mean(frac_list))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(scan_radii, mean_njets, "o-", color="crimson")
ax1.set_xlabel(r"Radius $R$")
ax1.set_ylabel(r"Mean Jet Count")
ax1.set_title(r"Jet Multiplicity vs $R$")

ax2.plot(scan_radii, mean_pt_frac, "s-", color="navy")
ax2.set_xlabel(r"Radius $R$")
ax2.set_ylabel(r"Leading Jet $p_T$ Fraction")
ax2.set_title(r"Energy Containment vs $R$")
plt.tight_layout()
plt.show()
```

<p><b>Physics Explanation</b>:<br>
1. <b>Multiplicity Drop</b>: As $R$ increases from 0.2 to 1.0, nearby small-$R$ jets are merged together into larger composite jets, steadily decreasing $\langle N_{\text{jets}} \rangle$.<br>
2. <b>Energy Containment Plateau</b>: For boosted $Z \to b\bar{b}$ decays with $p_T^Z > 150\text{ GeV}$, the opening angle $\Delta R_{b\bar{b}} \approx 2m_Z / p_T^Z \approx 0.8-1.0$. Once $R \ge 0.8$, the leading jet captures both $b$-quarks and their radiation, causing the energy fraction to plateau near $\sim 85-90\%$.</p>

</details>

---


> [!IMPORTANT]
> ### Self-Reflection & Physics Checkpoint 5.1
> **Conceptual Question**: What is one benefit and one cost of increasing the jet radius parameter $R$ from 0.4 to 1.0?

<details>
<summary>Click to show Checkpoint 5.1 Reference Solution</summary>

<ul>
<li><b>Benefit of $R=1.0$</b>: Captures all wide-angle decay products and gluon radiation from boosted heavy resonance decays (like $Z \to b\bar{b}$), reconstructing the full resonance mass inside a single large-$R$ jet.</li>
<li><b>Cost of $R=1.0$</b>: Collects significantly more soft background radiation, underlying event (UE) activity, and pileup interactions, which broadens jet energy resolution and increases susceptibility to background contamination.</li>
</ul>

</details>

---


## Step 6: Large-$R$ ($R=1.0$) Subjet Reclustering & Boosted $Z \to b\bar{b}$ Substructure

Now, let's explore **jet substructure**. When a high-$p_T$ $Z$ boson decays into a bottom-antibottom pair ($Z \to b\bar{b}$), both $b$-quarks fall inside a single $R=1.0$ large-$R$ jet.

To expose the internal two-prong decay structure:
1. Reconstruct large-$R$ jets with anti-$k_t$ ($R=1.0$).
2. Select the leading large-$R$ jet ($p_T > 100\text{ GeV}$).
3. Extract its constituents.
4. **Recluster** those constituents using exclusive $k_t$ algorithm with $N = 2$ subjets.
5. Compute subjet observables: subjet $p_T$, subjet mass, and subjet angular separation $\Delta R_{\text{subjet}}$.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 6.1: RECLUSTER LEADING LARGE-R JET INTO N=2 SUBJETS (KT ALGORITHM)
# ══════════════════════════════════════════════════════════════════════════════

subjet_results = []

for ev_idx, ev in enumerate(events_empflow):
    large_jets, _ = cluster_event(ev, algo_name="antikt", R=1.0, pt_min=100.0)
    if len(large_jets) == 0:
        continue
    
    lead_large_jet = large_jets[0]
    # Convert constituents tuple to list required by fastjet.ClusterSequence
    consts = list(lead_large_jet.constituents())
    
    subjet_def = fastjet.JetDefinition(fastjet.kt_algorithm, 1.0)
    sub_seq = fastjet.ClusterSequence(consts, subjet_def)
    exclusive_subjets = sub_seq.exclusive_jets(2)
    
    sorted_subjets = sorted(exclusive_subjets, key=lambda sj: sj.pt(), reverse=True)
    
    if len(sorted_subjets) == 2:
        sj1, sj2 = sorted_subjets[0], sorted_subjets[1]
        dr_sub = np.sqrt((sj1.eta() - sj2.eta())**2 + delta_phi(sj1.phi(), sj2.phi())**2)
        subjet_results.append({
            "event": ev_idx,
            "large_pt": lead_large_jet.pt(),
            "large_m": lead_large_jet.m(),
            "sj1_pt": sj1.pt(),
            "sj2_pt": sj2.pt(),
            "dr_sub": dr_sub
        })

print(f"Reclustered N=2 subjets (kt) for {len(subjet_results)} high-pT events.")
print(f"{'Event':<6s} | {'Large Jet pT':<13s} | {'Large Jet Mass':<14s} | {'Subjet 1 pT':<12s} | {'Subjet 2 pT':<12s} | {'Subjet dR':<10s}")
print("-" * 78)
for res in subjet_results:
    print(f"{res['event']:<6d} | {res['large_pt']:11.2f} GeV | {res['large_m']:12.2f} GeV | {res['sj1_pt']:10.2f} GeV | {res['sj2_pt']:10.2f} GeV | {res['dr_sub']:8.3f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 6.2: SUBJET KINEMATIC SPECTRA & THEORETICAL BOOST GUIDE
# ══════════════════════════════════════════════════════════════════════════════

if len(subjet_results) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    drs = [res["dr_sub"] for res in subjet_results]
    large_pts = [res["large_pt"] for res in subjet_results]
    
    ax1.scatter(large_pts, drs, color="crimson", s=60, edgecolors="k")
    pt_grid = np.linspace(100, 350, 100)
    dr_guide = 2.0 * 91.2 / pt_grid
    ax1.plot(pt_grid, dr_guide, "k--", label=r"Theoretical Boost Guide $\Delta R \approx 2 m_Z / p_T^Z$")
    
    ax1.set_xlabel(r"Large-$R$ Jet $p_T$ [GeV]", fontsize=10)
    ax1.set_ylabel(r"Subjet Angular Separation $\Delta R_{subjet}$", fontsize=10)
    ax1.set_title(r"Subjet Separation vs. Large-$R$ Transverse Momentum", fontsize=10, pad=8)
    ax1.grid(True, linestyle="--", alpha=0.4)
    ax1.legend(fontsize=9)

    sj1_pts = [res["sj1_pt"] for res in subjet_results]
    sj2_pts = [res["sj2_pt"] for res in subjet_results]
    ax2.hist(sj1_pts, bins=10, alpha=0.7, label=r"Leading Subjet $p_{T,1}$")
    ax2.hist(sj2_pts, bins=10, alpha=0.7, label=r"Subleading Subjet $p_{T,2}$")
    ax2.set_xlabel(r"Subjet Transverse Momentum [GeV]", fontsize=10)
    ax2.set_ylabel("Events", fontsize=10)
    ax2.set_title(r"Reclustered Subjet $p_T$ Spectra ($N=2$)", fontsize=10, pad=8)
    ax2.grid(True, linestyle="--", alpha=0.4)
    ax2.legend(fontsize=9)

    plt.tight_layout()
    plt.show()


---

> [!IMPORTANT]
> ### Exercise 6a: Subjet Momentum Sharing Fraction $z$
> **Task**: Write a Python script in the cell below to calculate the **subjet momentum sharing fraction** $z$ for each event with reclustered $N=2$ subjets:
> $$z = \frac{\min(p_{T,1}, p_{T,2})}{p_{T,1} + p_{T,2}}$$
> Plot a 1D histogram of $z$ across events.
>
> **Thought-Provoking Question**: For a hard two-body decay $Z \to b\bar{b}$, what range of $z$ values do you expect (symmetric vs. asymmetric)? How does soft QCD gluon radiation differ from a symmetric heavy particle decay in terms of $z$?


In [ ]:
# ── EXERCISE 6a: Compute Subjet Momentum Sharing Fraction z ────────────────────
z_values = []

# TODO 1: Calculate subjet momentum balance fraction z = min(pt1, pt2) / (pt1 + pt2) for each event in subjet_results
for res in subjet_results:
    pt1 = res["sj1_pt"]
    pt2 = res["sj2_pt"]
    z = ...  # <── TODO: Write formula min(pt1, pt2) / (pt1 + pt2) here!
    z_values.append(z)

# TODO 2: Plot 1D histogram of z_values with plt.hist()
fig, ax = plt.subplots(figsize=(8, 5))
# <── TODO: Write your histogram code here!

plt.tight_layout()
plt.show()


<details>
<summary>Click to show Exercise 6a Reference Solution & Explanation</summary>

```python
z_values = [min(r["sj1_pt"], r["sj2_pt"]) / (r["sj1_pt"] + r["sj2_pt"]) for r in subjet_results]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(z_values, bins=np.linspace(0, 0.5, 12), color="purple", alpha=0.75, edgecolor="k")
ax.set_xlabel(r"Subjet Momentum Balance $z$")
ax.set_ylabel("Events")
ax.set_title(r"Subjet Momentum Balance Spectrum")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
```

<p><b>Physics Explanation</b>:<br>
1. <b>Symmetric Resonance Decays</b>: In a two-body decay of a heavy scalar/vector resonance ($Z \to b\bar{b}$), the decay partons share momentum nearly equally in the rest frame, yielding $z \approx 0.3 - 0.5$ (symmetric splitting).<br>
2. <b>Soft QCD Gluon Emission</b>: In contrast, soft gluon bremsstrahlung $q \to qg$ is dominated by the soft singularity $d\sigma / dz \sim 1/z$, producing highly asymmetric momentum sharing ($z \to 0$). Heavy resonance taggers rely on cuts like $z > 0.1$ (or Soft Drop grooming) to reject soft QCD backgrounds.</p>

</details>

---


> [!IMPORTANT]
> ### Exercise 6b: Cambridge/Aachen (C/A) Subjet Reclustering Comparison
> **Task**: Write Python code in the cell below to recluster the constituents of the leading $R=1.0$ large-$R$ jet using exclusive **Cambridge/Aachen (C/A)** algorithm (`fastjet.cambridge_algorithm`) with $N=2$ subjets:
> 1. Recluster constituents with exclusive C/A ($N=2$).
> 2. Compute subjet angular separation $\Delta R_{\text{subjet}}^{\text{C/A}}$ and subjet momentum balance $z^{\text{C/A}}$.
> 3. Plot side-by-side comparison histograms of $\Delta R_{\text{subjet}}$ and $z$ comparing $k_t$ reclustering vs. C/A reclustering across all events.
>
> **Thought-Provoking Physics Question**: Why is Cambridge/Aachen ($p=0$, pure angular distance ordering) uniquely suited for declustering jet clustering trees backwards from wide to narrow angles in jet grooming algorithms like Soft Drop or Mass Drop Tagger?


In [ ]:
# ── EXERCISE 6b: Exclusive C/A Subjet Reclustering & Comparison ───────────────
ca_drs = []
ca_zs = []
kt_drs = [res["dr_sub"] for res in subjet_results]
kt_zs = [min(res["sj1_pt"], res["sj2_pt"]) / (res["sj1_pt"] + res["sj2_pt"]) for res in subjet_results]

# TODO 1: Recluster constituents of leading large-R jet using exclusive Cambridge/Aachen algorithm (N=2)
# Hint: ca_def = fastjet.JetDefinition(fastjet.cambridge_algorithm, 1.0)
# Hint: ca_seq = fastjet.ClusterSequence(consts, ca_def)
# Hint: ca_subjets = sorted(ca_seq.exclusive_jets(2), key=lambda sj: sj.pt(), reverse=True)
for ev in events_empflow:
    # <── TODO: Write your C/A reclustering loop here!
    pass

# TODO 2: Plot side-by-side comparison histograms comparing kt subjets vs C/A subjets
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# <── TODO: Write your comparison histogram code here!

plt.tight_layout()
plt.show()


<details>
<summary>Click to show Exercise 6b Reference Solution & Explanation</summary>

```python
ca_drs, ca_zs = [], []
for ev in events_empflow:
    large_jets, _ = cluster_event(ev, algo_name="antikt", R=1.0, pt_min=100.0)
    if len(large_jets) == 0: continue
    consts = list(large_jets[0].constituents())
    ca_seq = fastjet.ClusterSequence(consts, fastjet.JetDefinition(fastjet.cambridge_algorithm, 1.0))
    subs = sorted(ca_seq.exclusive_jets(2), key=lambda sj: sj.pt(), reverse=True)
    if len(subs) == 2:
        ca_drs.append(np.sqrt((subs[0].eta() - subs[1].eta())**2 + delta_phi(subs[0].phi(), subs[1].phi())**2))
        ca_zs.append(min(subs[0].pt(), subs[1].pt()) / (subs[0].pt() + subs[1].pt()))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(kt_drs, bins=10, alpha=0.7, label=r"$k_t$")
ax1.hist(ca_drs, bins=10, alpha=0.7, label=r"C/A")
ax1.legend()
ax2.hist(kt_zs, bins=10, alpha=0.7, label=r"$k_t$")
ax2.hist(ca_zs, bins=10, alpha=0.7, label=r"C/A")
ax2.legend()
plt.tight_layout()
plt.show()
```

<p><b>Physics Explanation</b>:<br>
Because Cambridge/Aachen ($p=0$) clusters strictly according to angular separation $\Delta R_{ij}^2/R^2$, undoing the clustering sequence step-by-step (declustering) undoes the angular-ordered parton shower from widest angles to narrowest angles. This property makes C/A declustering the foundation of jet grooming algorithms like Soft Drop, which uncluster the C/A tree until finding a hard two-prong splitting ($z > z_{\text{cut}}$) that resolves the resonance core.</p>

</details>

---


> [!IMPORTANT]
> ### Self-Reflection & Physics Checkpoint 6.1
> **Conceptual Question**: Why does observing two subjets inside a large-$R$ jet not establish by itself that the jet originated from a $Z \to b\bar{b}$ decay?

<details>
<summary>Click to show Checkpoint 6.1 Reference Solution</summary>

<p>Reclustering a large-$R$ jet into $N=2$ subjets simply enforces a two-body momentum division. Pure QCD background jets produced by gluon splitting ($g \to q\bar{q}$) or hard asymmetric gluon radiation also exhibit two-prong substructure. Confirming a true $Z \to b\bar{b}$ decay requires invariant mass consistency ($m_{\text{jet}} \approx m_Z = 91.2\text{ GeV}$) combined with $b$-tagging algorithms to verify bottom-quark flavor origin.</p>

</details>

---


## Step 7: Truth-Level vs. EMPFlow Detector-Level Comparison

Finally, let's compare **truth-level particles** (`constituents_Truth_*`) against **detector-level EMPFlow objects** (`constituents_EMPFlow_*`).

This comparison illustrates how hadronization, neutral particle loss, and calorimeter cluster merging alter constituent multiplicities, jet energy scales, and reconstructed jet masses.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 7.1: LOAD TRUTH DATASET & CLUSTER TRUTH VS. EMPFLOW LARGE-R JETS
# ══════════════════════════════════════════════════════════════════════════════

def load_truth_dataset(filepath):
    """
    Loads generator-level truth particles (constituents_Truth_*) from ROOT tree 'analysis'.
    Raises FileNotFoundError if dataset file does not exist.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Truth dataset file '{filepath}' not found. Please verify data path!")

    with uproot.open(filepath) as f:
        tree = f["analysis"]
        arrays = tree.arrays([
            "constituents_Truth_pt", "constituents_Truth_eta",
            "constituents_Truth_phi", "constituents_Truth_m",
            "constituents_Truth_e", "constituents_Truth_pdgId",
            "constituents_Truth_charge"
        ], library="ak")
        
        events = []
        for i in range(len(arrays)):
            pt_raw = ak.to_numpy(arrays["constituents_Truth_pt"][i])
            m_raw = ak.to_numpy(arrays["constituents_Truth_m"][i])
            e_raw = ak.to_numpy(arrays["constituents_Truth_e"][i])
            scale = 1000.0 if np.mean(pt_raw) > 500 else 1.0
            
            events.append({
                "pt": pt_raw / scale,
                "eta": ak.to_numpy(arrays["constituents_Truth_eta"][i]),
                "phi": ak.to_numpy(arrays["constituents_Truth_phi"][i]),
                "m": m_raw / scale,
                "e": e_raw / scale,
                "pdgId": ak.to_numpy(arrays["constituents_Truth_pdgId"][i]),
                "charge": ak.to_numpy(arrays["constituents_Truth_charge"][i])
            })
        return events

events_truth = load_truth_dataset(dataset_file)

truth_nconst = [len(ev["pt"]) for ev in events_truth]
emp_nconst = [len(ev["pt"]) for ev in events_empflow]

truth_jet_masses = []
emp_jet_masses = []

for i in range(len(events_truth)):
    j_t, _ = cluster_event(events_truth[i], algo_name="antikt", R=1.0, pt_min=50.0)
    j_e, _ = cluster_event(events_empflow[i], algo_name="antikt", R=1.0, pt_min=50.0)
    if len(j_t) > 0 and len(j_e) > 0:
        truth_jet_masses.append(j_t[0].m())
        emp_jet_masses.append(j_e[0].m())

print("=== Truth vs EMPFlow Comparison Summary ===")
print(f"Mean Truth Constituent Count : {np.mean(truth_nconst):.1f}")
print(f"Mean EMPFlow Constituent Count: {np.mean(emp_nconst):.1f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 7.2: PLOT CONSTITUENT MULTIPLICITY & JET MASS: TRUTH VS. EMPFLOW
# ══════════════════════════════════════════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(truth_nconst, bins=10, alpha=0.7, label="Truth Particles", color="forestgreen")
ax1.hist(emp_nconst, bins=10, alpha=0.7, label="EMPFlow Detector Objects", color="navy")
ax1.set_xlabel(r"Event Constituent Multiplicity $N_{const}$", fontsize=10)
ax1.set_ylabel("Events", fontsize=10)
ax1.set_title("Constituent Multiplicity: Truth vs. EMPFlow", fontsize=10, pad=8)
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend(fontsize=9)

if len(truth_jet_masses) > 0:
    ax2.hist(truth_jet_masses, bins=8, alpha=0.7, label=r"Truth Large-$R$ Jet Mass", color="forestgreen")
    ax2.hist(emp_jet_masses, bins=8, alpha=0.7, label=r"EMPFlow Large-$R$ Jet Mass", color="navy")
    ax2.set_xlabel(r"Leading Large-$R$ Jet Mass $m_{jet}$ [GeV]", fontsize=10)
    ax2.set_ylabel("Events", fontsize=10)
    ax2.set_title(r"Reconstructed Jet Mass: Truth vs. EMPFlow ($R=1.0$)", fontsize=10, pad=8)
    ax2.grid(True, linestyle="--", alpha=0.4)
    ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()


---

> [!IMPORTANT]
> ### Exercise 7a: Jet Energy Scale & Mass Response Ratio ($R_m$)
> **Task**: Write Python code in the cell below to compute the event-by-event reconstructed jet mass response ratio:
> $$R_m = \frac{m_{\text{jet}}^{\text{EMPFlow}}}{m_{\text{jet}}^{\text{Truth}}}$$
> for leading $R=1.0$ large-$R$ jets, and plot a 1D histogram of $R_m$.
>
> **Thought-Provoking Question**: Is $R_m$ systematically less than 1 or greater than 1? What physical detector effects (e.g., out-of-cone energy loss, tracking thresholds, calorimeter cluster merging) explain why detector-level EMPFlow jet mass differs from generator truth particle jet mass?


In [ ]:
# ── EXERCISE 7a: Jet Mass Response Ratio Rm = m_EMPFlow / m_Truth ─────────────
rm_ratios = []

# TODO 1: Cluster R=1.0 anti-kt jets on Truth and EMPFlow collections and compute ratio m_emp / m_truth
# Hint: j_t, _ = cluster_event(events_truth[i], algo_name="antikt", R=1.0, pt_min=50.0)
# Hint: j_e, _ = cluster_event(events_empflow[i], algo_name="antikt", R=1.0, pt_min=50.0)
for i in range(min(len(events_truth), len(events_empflow))):
    # <── TODO: Write your response ratio loop here!
    pass

# TODO 2: Plot 1D histogram of rm_ratios with reference line at 1.0 (axvline)
fig, ax = plt.subplots(figsize=(8, 5))
# <── TODO: Write your histogram code here!

plt.tight_layout()
plt.show()


<details>
<summary>Click to show Exercise 7a Reference Solution & Explanation</summary>

```python
rm_ratios = []
for i in range(min(len(events_truth), len(events_empflow))):
    j_t, _ = cluster_event(events_truth[i], algo_name="antikt", R=1.0, pt_min=50.0)
    j_e, _ = cluster_event(events_empflow[i], algo_name="antikt", R=1.0, pt_min=50.0)
    if len(j_t) > 0 and len(j_e) > 0:
        rm_ratios.append(j_e[0].m() / j_t[0].m())

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(rm_ratios, bins=12, color="darkorange", alpha=0.8, edgecolor="k")
ax.axvline(1.0, color="k", linestyle="--")
ax.set_xlabel(r"Jet Mass Response Ratio $R_m$")
ax.set_ylabel("Events")
ax.set_title("Jet Mass Response Spectrum")
plt.tight_layout()
plt.show()
```

<p><b>Physics Explanation</b>:<br>
1. <b>Detector Energy Loss & Response</b>: In uncalibrated detector-level EMPFlow objects, low-energy neutral particles below calorimeter cluster thresholds or charged hadrons outside inner detector tracking acceptance are missed, causing $R_m < 1.0$ (typically $\sim 0.85-0.95$).<br>
2. <b>Energy Scale Calibration</b>: Experimental analyses apply Jet Energy Scale (JES) and Jet Mass Scale (JMS) calibrations to restore the mean response to $R_m = 1.0$.</p>

</details>

---


## Summary & Next Steps

Congratulations! You have completed **Part 2: Jet Clustering, Radius Dependence, & Subjet Exploration**.

### Key Concepts & Exercises Mastered:
1. **Physics Foundations First**: Analyzed color confinement, parton showering, and sequential recombination algorithm distance metrics ($d_{ij}, d_{iB}$).
2. **Environment & Data Setup**: Configured persistent storage and loaded detector-level EMPFlow and truth-level constituent arrays.
3. **Interactive Constituent Analysis**: Plotted constituent $p_T$ and $\eta$ spectra to understand soft QCD particle production.
4. **Pure-Python & FastJet Clustering**: Built a pure-Python anti-$k_t$ algorithm from scratch and verified identical performance against FastJet C++ bindings.
5. **Complete $(\eta, \phi)$ Event Display Mapping**: Visualized all event constituents, highlighted constituent kinematic selections, marked jet center axes (`X`), and drew jet boundary acceptance circles ($R$).
6. **Radius Parameter Scan**: Evaluated jet multiplicity drop and energy containment plateau across $R \in \{0.2, 0.4, 0.6, 0.8, 1.0\}$.
7. **Subjet Substructure & Balance**: Reclustered $R=1.0$ large-$R$ jets into $N=2$ subjets using both $k_t$ and Cambridge/Aachen (C/A) algorithms, comparing $\Delta R_{\text{subjet}}$ and momentum balance $z$.
8. **Truth vs. Detector Response**: Computed jet mass response $R_m$ to quantify detector resolution shifts.

**Proceed to Part 3 (Detector-Level ROOT Analysis, Machine Learning Taggers, & MC Normalization)** to apply machine learning jet taggers and perform detector-level ROOT analysis!
